In [1]:
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Load Required Datasets
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")


## Collaborative Filtering Recommender System ##
---
---
- **ONLY USER-ITEM INTERACTIONS**
---
- 1️⃣ User-based CF → 
    - Predicts ratings for a user based on ratings of similar users.
    - Relies on user-user similarity
---
- 2️⃣ Item-based CF →
    - Predicts ratings for a user based on similar items they have already rated.
    - Relies on item-item similarity.
---
- 3️⃣ Model-Based Collaborative Filtering →
    - Uses matrix factorization or machine learning models to learn latent features.
    - Example: SVD, ALS, or Neural Collaborative Filtering.
    - Learns hidden features of users and items to predict ratings.
---
---

### 1️⃣ User-Based Collaborative Filtering ###

In [3]:
# User-item matrix
user_item = ratings.pivot(index="userId", columns="movieId", values="rating").fillna(0)

# User similarity
user_sim = cosine_similarity(user_item)
user_sim = pd.DataFrame(user_sim, index=user_item.index, columns=user_item.index)

# User-based Recommendation function
def user_based_recs(userId, top_n=10):
    pred_ratings = user_sim.loc[userId] @ user_item / user_sim.loc[userId].sum()
    pred_ratings = pd.Series(pred_ratings, index=user_item.columns)
    
    rated_movies = ratings[ratings["userId"] == userId]["movieId"].tolist()
    pred_ratings = pred_ratings.drop(rated_movies, errors="ignore")
    
    top_movies = pred_ratings.sort_values(ascending=False).head(top_n)
    return pd.DataFrame({
        "title": top_movies.index.map(lambda m: movies.loc[movies["movieId"]==m, "title"].values[0]),
        "pred_rating": top_movies.values
    })


print("Top User-Based Recommendations for User 1:")
print(user_based_recs(1))


Top User-Based Recommendations for User 1:
                                               title  pred_rating
0                   Shawshank Redemption, The (1994)     2.622414
1                  Terminator 2: Judgment Day (1991)     2.061920
2                              Godfather, The (1972)     1.836914
3                            Sixth Sense, The (1999)     1.643315
4  Lord of the Rings: The Fellowship of the Ring,...     1.605043
5                                   Apollo 13 (1995)     1.566524
6          Twelve Monkeys (a.k.a. 12 Monkeys) (1995)     1.564531
7  Lord of the Rings: The Return of the King, The...     1.483950
8      Lord of the Rings: The Two Towers, The (2002)     1.465234
9                                     Aladdin (1992)     1.459054


### 2️⃣ Item-Based Collaborative Filtering ###

In [4]:
# Create user-item matrix
user_item = ratings.pivot(index="userId", columns="movieId", values="rating").fillna(0)

# Create item-user matrix
item_user = user_item.T

# Compute item-item similarity
item_sim = cosine_similarity(item_user)
item_sim = pd.DataFrame(item_sim, index=item_user.index, columns=item_user.index)

# Item-based Recommendation function
def item_based_recs(userId, top_n=10):
    user_ratings = user_item.loc[userId]  # ratings for this user
    unrated_movies = user_ratings[user_ratings==0].index

    pred_ratings = {}
    for movie in unrated_movies:
        rated_items = user_ratings[user_ratings>0].index
        if len(rated_items) == 0:
            pred_ratings[movie] = 0
            continue
        sim_scores = item_sim.loc[movie, rated_items]  # similarity only for rated items
        ratings = user_ratings[rated_items]
        numerator = (sim_scores * ratings).sum()
        denominator = sim_scores.abs().sum()
        pred_ratings[movie] = numerator / (denominator + 1e-8)

    pred_ratings = pd.Series(pred_ratings).sort_values(ascending=False).head(top_n)
    return pd.DataFrame({
        "title": pred_ratings.index.map(lambda m: movies.loc[movies["movieId"]==m, "title"].values[0]),
        "pred_rating": pred_ratings.values
    })


print("Top Item-Based Recommendations for User 1:")
print(item_based_recs(1))


Top Item-Based Recommendations for User 1:
                                               title  pred_rating
0  The Jinx: The Life and Deaths of Robert Durst ...     5.000000
1                                      Circus (2000)     5.000000
2                               Alvarez Kelly (1966)     5.000000
3                                   War Dance (2007)     4.999999
4                        Harlan County U.S.A. (1976)     4.999999
5              Devil and Daniel Johnston, The (2005)     4.999999
6                                 The Clapper (2018)     4.999999
7  Fireworks, Should We See It from the Side or t...     4.999999
8          Black Butler: Book of the Atlantic (2017)     4.999999
9              The Stanford Prison Experiment (2015)     4.999999


### 3️⃣ Model-Based Collaborative Filtering (Matrix Factorization) ###

In [5]:
# Re-map userId and movieId to continuous indices
user_map = {u: i for i, u in enumerate(ratings["userId"].unique())}
movie_map = {m: i for i, m in enumerate(movies["movieId"].unique())}

ratings["user_idx"] = ratings["userId"].map(user_map)
ratings["movie_idx"] = ratings["movieId"].map(movie_map)

num_users = len(user_map)
num_movies = len(movie_map)
num_features = 10                                                          # latent dimensions

print(f"Users: {num_users}, Movies: {num_movies}")


# Create Y (ratings) and R (mask)
Y = np.zeros((num_movies, num_users))
R = np.zeros((num_movies, num_users))

for row in ratings.itertuples(index=False):
    Y[row.movie_idx, row.user_idx] = row.rating
    R[row.movie_idx, row.user_idx] = 1

print("Y shape:", Y.shape, "| R shape:", R.shape)


# Initialize parameters
np.random.seed(42)
X = np.random.randn(num_movies, num_features) * 0.01
W = np.random.randn(num_users, num_features) * 0.01
b = np.zeros((1, num_users))


# Cost function + gradients
def cofi_cost_grad(X, W, b, Y, R, lambda_):
    y_pred = X @ W.T + b  # (movies × users)
    error = R * (y_pred - Y)

    # Cost
    J = 0.5 * np.sum(error**2) + (lambda_/2) * (np.sum(X**2) + np.sum(W**2))

    # Gradients
    X_grad = error @ W + lambda_ * X
    W_grad = error.T @ X + lambda_ * W
    b_grad = np.sum(error, axis=0, keepdims=True)

    return J, X_grad, W_grad, b_grad


# Gradient Descent Training
alpha = 0.0001   # learning rate
lambda_ = 0.5    # regularization
epochs = 200

for epoch in range(1,epochs+1):
    J, X_grad, W_grad, b_grad = cofi_cost_grad(X, W, b, Y, R, lambda_)

    # Update parameters
    X -= alpha * X_grad
    W -= alpha * W_grad
    b -= alpha * b_grad

    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d}: Cost = {J:.4f}")


# Model-based Recommendation function
def model_based_recs(userId, top_n=10):
    """
    Recommend top_n movies for a given userId (original MovieLens ID).
    """
    if userId not in user_map:
        print("User not found.")
        return []

    user_idx = user_map[userId]
    y_pred = X @ W.T + b  # predicted ratings
    user_ratings = y_pred[:, user_idx]

    # Exclude already rated movies
    rated_movies = ratings.loc[ratings["userId"] == userId, "movieId"].tolist()
    unrated_mask = ~movies["movieId"].isin(rated_movies)

    movie_scores = pd.DataFrame({
        "movieId": movies["movieId"],
        "title": movies["title"],
        "pred_rating": user_ratings
    })

    recs = movie_scores[unrated_mask].sort_values("pred_rating", ascending=False).head(top_n)
    return recs[["title", "pred_rating"]]


print("\nTop Model-Based Recommendations for User 1:")
print(model_based_recs(1))

Users: 610, Movies: 9742
Y shape: (9742, 610) | R shape: (9742, 610)
Epoch 020: Cost = 263769.1322
Epoch 040: Cost = 173031.0560
Epoch 060: Cost = 132219.0965
Epoch 080: Cost = 108974.8567
Epoch 100: Cost = 94044.3315
Epoch 120: Cost = 83717.8404
Epoch 140: Cost = 76206.6286
Epoch 160: Cost = 70539.6253
Epoch 180: Cost = 66143.8068
Epoch 200: Cost = 62659.1319

Top Model-Based Recommendations for User 1:
                                              title  pred_rating
4137  Lord of the Rings: The Two Towers, The (2002)     4.327795
6710                        Dark Knight, The (2008)     4.327703
277                Shawshank Redemption, The (1994)     4.327684
474                             Blade Runner (1982)     4.327653
3568                          Monsters, Inc. (2001)     4.327590
6772                                  WALL·E (2008)     4.327469
8666                              It Follows (2014)     4.327402
3180                      Frankie and Johnny (1991)     4.327399
5321   